In [4]:
# === All-in-one: Arabic Legal Text Preprocessing (for Laws & Regulations) ===
# File: laws_and_regulations_preprocessing.py
# Author: Lujain
# Purpose: Clean and segment Arabic legal documents (laws, regulations)
# Input: Government_Tenders_and_Procurement_Law.txt
# Output: *_clean.txt and *_articles.json in same folder

import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple
import json

# ============================================================
# 1) Unicode normalization
# ============================================================

def normalize_unicode(text: str) -> str:
    """Normalize Unicode to NFC for consistent Arabic representation."""
    return unicodedata.normalize("NFC", text)


# ============================================================
# 2) Remove Arabic diacritics (tashkeel + Quranic marks)
# ============================================================

ARABIC_DIACRITICS_RE = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]')

def remove_diacritics(text: str) -> str:
    """Remove Arabic diacritics and Quranic marks."""
    return ARABIC_DIACRITICS_RE.sub("", text)


# ============================================================
# 3) Combined character normalization (tatweel + letters + digits)
# ============================================================

COMBINED_TRANSLATION_MAP = {
    ord("\u0640"): None,  # Tatweel (ـ)
    # Normalize alif forms
    ord("\u0622"): ord("\u0627"),  # آ  → ا
    ord("\u0623"): ord("\u0627"),  # أ  → ا
    ord("\u0625"): ord("\u0627"),  # إ  → ا
    ord("\u0671"): ord("\u0627"),  # ٱ  → ا
    # Normalize alef maqsura
    ord("\u0649"): ord("\u064A"),  # ى → ي
    # Arabic-Indic digits → Western digits
    ord("٠"): ord("0"),
    ord("١"): ord("1"),
    ord("٢"): ord("2"),
    ord("٣"): ord("3"),
    ord("٤"): ord("4"),
    ord("٥"): ord("5"),
    ord("٦"): ord("6"),
    ord("٧"): ord("7"),
    ord("٨"): ord("8"),
    ord("٩"): ord("9"),
}

def normalize_chars(text: str) -> str:
    """Normalize tatweel, alif variants, alef maqsura, and digits in one pass."""
    return text.translate(COMBINED_TRANSLATION_MAP)


# ============================================================
# 4) Standardize bullets and punctuation
# ============================================================

BULLET_CHARS_RE = re.compile(r"[•●◦▪\-–—]+")

def standardize_bullets_and_punct(text: str) -> str:
    """Normalize various bullet-like characters to a single '-'."""
    return BULLET_CHARS_RE.sub("-", text)


# ============================================================
# 5) Clean whitespace (spaces + blank lines)
# ============================================================

def clean_whitespace(text: str) -> str:
    """
    Collapse multiple spaces and empty lines.
    """
    lines = text.splitlines()
    cleaned_lines = []
    prev_blank = False

    for line in lines:
        line_clean = " ".join(line.split()).strip()
        if line_clean == "":
            if not prev_blank:
                cleaned_lines.append("")
            prev_blank = True
        else:
            cleaned_lines.append(line_clean)
            prev_blank = False
    return "\n".join(cleaned_lines)


# ============================================================
# 6) Article detection & segmentation (النظام / اللائحة)
# ============================================================

ARABIC_ORDINAL_MAP = {
    "الاولى": 1, "الأولى": 1, "الثانية": 2, "الثالثة": 3, "الرابعة": 4,
    "الخامسة": 5, "السادسة": 6, "السابعة": 7, "الثامنة": 8, "التاسعة": 9, "العاشرة": 10,
}

ARTICLE_HEADER_RE = re.compile(
    r"""
    (?P<source>
        النظام
        |
        ال(?:لائحة|لائحه|ائحه)
    )
    \s*[:：]?\s*
    (?:المادة|الماده)
    \s*
    (?P<num>[\d]+|[^\s:]+)
    """,
    re.VERBOSE
)

def normalize_article_number(num_str: str) -> str:
    """Normalize article numbering."""
    num_str = num_str.strip()
    if num_str.isdigit():
        return num_str
    if num_str in ARABIC_ORDINAL_MAP:
        return str(ARABIC_ORDINAL_MAP[num_str])
    return num_str

def normalize_source_label(source_raw: str) -> str:
    """Normalize source to either 'النظام' or 'اللائحة'."""
    return "النظام" if "نظام" in source_raw else "اللائحة"

def segment_articles(text: str) -> List[Dict]:
    """
    Segment text into articles like:
    - النظام: الماده 5
    - اللائحة:الماده134
    """
    articles: List[Dict] = []
    matches = list(ARTICLE_HEADER_RE.finditer(text))
    if not matches:
        return articles

    for idx, match in enumerate(matches):
        start = match.start()
        end = match.end()
        source_raw = match.group("source")
        source = normalize_source_label(source_raw)
        raw_num = match.group("num")
        article_no = normalize_article_number(raw_num)

        if idx + 1 < len(matches):
            next_start = matches[idx + 1].start()
            article_body = text[start:next_start]
        else:
            article_body = text[start:]

        articles.append({
            "source": source,
            "article_no": article_no,
            "header_span": [start, end],
            "text": article_body.strip(),
        })
    return articles


# ============================================================
# 7) Full preprocessing pipeline
# ============================================================

def preprocess_text(text: str) -> Tuple[str, List[Dict]]:
    """Run the full preprocessing pipeline on the text."""
    text = normalize_unicode(text)
    text = remove_diacritics(text)
    text = normalize_chars(text)
    text = standardize_bullets_and_punct(text)
    text = clean_whitespace(text)
    articles = segment_articles(text)
    return text, articles


# ============================================================
# 8) Local I/O (VS Code / normal Python)
# ============================================================

# Input file path (same directory)
input_path = Path("Government_Tenders_and_Procurement_Law.txt")

# Output file paths
clean_txt_path = input_path.with_name("Government_Tenders_and_Procurement_Law_clean.txt")
articles_json_path = input_path.with_name("Government_Tenders_and_Procurement_Law_articles.json")

print(f"Reading input file from: {input_path}")

# Read and preprocess
raw_text = input_path.read_text(encoding="utf-8")
cleaned_text, articles = preprocess_text(raw_text)

# Save results
clean_txt_path.write_text(cleaned_text, encoding="utf-8")
articles_json_path.write_text(json.dumps(articles, ensure_ascii=False, indent=2), encoding="utf-8")

print(f" Saved cleaned text to: {clean_txt_path}")
print(f" Saved articles JSON to: {articles_json_path}")
print(f" Number of detected articles: {len(articles)}")


Reading input file from: Government_Tenders_and_Procurement_Law.txt
 Saved cleaned text to: Government_Tenders_and_Procurement_Law_clean.txt
 Saved articles JSON to: Government_Tenders_and_Procurement_Law_articles.json
 Number of detected articles: 354


In [ ]:
import json
from pathlib import Path

articles_json_path = Path("Government_Tenders_and_Procurement_Law_articles.json")

with open(articles_json_path, "r", encoding="utf-8") as f:
    articles = json.load(f)

print(f"✅ Loaded {len(articles)} articles from {articles_json_path}")


✅ Loaded 354 articles from Government_Tenders_and_Procurement_Law_articles.json


In [22]:
# ============================================================
# 9) Search & Query Helper Functions
# ============================================================

def get_articles_by_number(source: str, article_no: str, articles: List[Dict]) -> List[Dict]:
    """
    Search by (source, article number).
    Example: get_articles_by_number("النظام", "76", articles)
    """
    return [
        art for art in articles
        if art["source"] == source and art["article_no"] == article_no
    ]


def search_articles_by_keyword(keyword: str, articles: List[Dict]) -> List[Dict]:
    """
    Search for a keyword inside all article texts (case-sensitive).
    Example: search_articles_by_keyword("انهاء العقد", articles)
    """
    results = []
    for art in articles:
        if keyword in art["text"]:
            results.append(art)
    return results


def print_article(art: Dict):
    """Pretty-print one article."""
    print("==========================================================")
    print(f"📘 Source     : {art['source']}")
    print(f"📑 Article No : {art['article_no']}")
    print("----------------------------------------------------------")
    print(art["text"][:1000])  # print first 1000 chars
    print("==========================================================\n")


# ============================================================
# 10) Simple interactive query mode
# ============================================================

if __name__ == "__main__":
    print("\n🔎 Legal Text Query Mode 🔍")
    print("Choose one of the following:")
    print("1) Search by article number (source + number)")
    print("2) Search by keyword")
    print("0) Exit\n")

    while True:
        choice = input("Enter your choice (0-2): ").strip()

        if choice == "0":
            print("Exiting search mode.")
            break

        elif choice == "1":
            src = input("Enter source (النظام / اللائحة): ").strip()
            num = input("Enter article number (e.g., 76, 133, ...): ").strip()
            matches = get_articles_by_number(src, num, articles)

            if not matches:
                print("❌ No matching article found.\n")
            else:
                for art in matches:
                    print_article(art)

        elif choice == "2":
            kw = input("Enter keyword to search for: ").strip()
            matches = search_articles_by_keyword(kw, articles)

            if not matches:
                print(f"❌ No results for '{kw}'\n")
            else:
                print(f"✅ Found {len(matches)} result(s) containing '{kw}':\n")
                for art in matches:
                    print_article(art)

        else:
            print("⚠️ Invalid choice, please try again.\n")



🔎 Legal Text Query Mode 🔍
Choose one of the following:
1) Search by article number (source + number)
2) Search by keyword
0) Exit

✅ Found 3 result(s) containing 'يجب علي الجهة الحكومية انهاء العقد في الحالات الاتية':

📘 Source     : النظام
📑 Article No : 76
----------------------------------------------------------
النظام:الماده76

.1 يجب علي الجهة الحكومية انهاء العقد في الحالات الاتية:
ا-اذا تبين ان المتعاقد معه قد شرع -بنفسه او بوساطة غيره بطريق مباشر او غير مباشر- في رشوة احد موظفي الجهات الخاضعة لاحكام النظام او حصل علي العقد عن طريق الرشوة او الغش او التحايل او التزوير او التلاعب او مارس ايا من ذلك اثناء تنفيذه للعقد.
ب-اذا افلس المتعاقد معه، او طلب اشهار افلاسه، او ثبت اعساره، او صدر امر بوضعه تحت الحراسة، او كان شركة وجري حلها او تصفيتها.
ج- اذا تنازل المتعاقد معه عن العقد دون موافقة مكتوبة من الجهة الحكومية والوزارة. .2 يجوز للجهة الحكومية انهاء العقد في الحالات التالية:
ا- اذا تاخر المتعاقد معه عن البدء في العمل, او تباطا في تنفيذه, او اخل باي شرط من شروط العقد ولم يصحح اوض